# Fine-Tuning of LLMs with Hugging Face


## Step 1: Installing and importing the libraries for Hugging Face


In [1]:
import os

# Disable external experiment logging for classroom/Colab runs.
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
# Runtime > Restart runtime after this cell if Colab asks you to.
!pip install -q -U "transformers>=4.57.1,<5.0" "datasets>=3.0" "accelerate>=1.0" "peft>=0.13" "trl>=0.24" "bitsandbytes>=0.44" "huggingface_hub>=0.26"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.4 MB/s eta 0:00:00


In [3]:
# Optional: check installed versions.
import transformers, datasets, peft, trl, accelerate

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.57.6
datasets: 4.8.5
peft: 0.19.1
trl: 1.4.0
accelerate: 1.13.0


In [4]:
import os
import torch

from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
    logging,
)
from trl import SFTConfig, SFTTrainer

logging.set_verbosity_error()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


## Step 2: Setting up links to Hugging Face datasets and models


In [5]:
model_identifier = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
source_dataset = "gamino/wiki_medical_terms"
formatted_dataset = "aboonaji/wiki_medical_terms_llam2_format"


## Step 3: Setting up all the QLoRA hyperparameters for fine-tuning

In [6]:

lora_hyper_r = 16
lora_hyper_alpha = 32
lora_hyper_dropout = 0.05

## Step 4: Setting up all the bitsandbytes hyperparameters for fine-tuning

In [7]:
enable_4bit = torch.cuda.is_available()
compute_dtype_bnb = "bfloat16"
quant_type_bnb = "nf4"
double_quant_flag = False

print("4-bit QLoRA enabled:", enable_4bit)

4-bit QLoRA enabled: True


## Step 5: Setting up all the training arguments hyperparameters for fine-tuning

In [8]:
results_dir = "./results"
epochs_count = 1

train_batch_size = 1
eval_batch_size = 1
accumulation_steps = 4
steps_limit = 30

checkpointing_flag = torch.cuda.is_available()
grad_norm_limit = 0.3
train_learning_rate = 2e-4
decay_rate = 0.001
optimizer_type = "paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch"
scheduler_type = "cosine"
warmup_percentage = 0.03
length_grouping = True
log_interval = 5

enable_fp16 = False
enable_bf16 = torch.cuda.is_available()

## Step 6: Setting up all the supervised fine-tuning arguments hyperparameters for fine-tuning

In [9]:
enable_packing = False
sequence_length_max = 512

device_assignment = "auto" if torch.cuda.is_available() else None


## Step 7: Loading the dataset

In [10]:
training_data = load_dataset(formatted_dataset, split="train")

print(training_data)
print("Columns:", training_data.column_names)
print("First example:")
print(training_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


wiki_medical_terms_llam2.jsonl:   0%|          | 0.00/54.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6861 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 6861
})
Columns: ['text']
First example:
{'text': '<s> [INST] <<SYS>> You are a helpful, respectful, and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature. If a question does not make any sense or is not factually coherent, explain why instead of answering something not correct. If you don\'t know the answer to a question, please don\'t share false information. <</SYS>> What is Paracetamol poisoning and explain in detail? [/INST] Paracetamol poisoning, also known as acetaminophen poisoning, is caused by excessive use of the medication paracetamol (acetaminophen). Most people have few or non-specific symptoms in the first 24 hours following overdose. These include feeling tired, abdominal pain, or nausea. This is typically follo

In [11]:
# Make sure the dataset has the text column expected by SFTTrainer.
# The selected formatted dataset normally already contains a column named "text".

if "text" not in training_data.column_names:
    raise ValueError(
        f"Expected a column named 'text', but found: {training_data.column_names}. "
        "Update the dataset formatting step before training."
    )

print(training_data[0]["text"][:1000])

<s> [INST] <<SYS>> You are a helpful, respectful, and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature. If a question does not make any sense or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share false information. <</SYS>> What is Paracetamol poisoning and explain in detail? [/INST] Paracetamol poisoning, also known as acetaminophen poisoning, is caused by excessive use of the medication paracetamol (acetaminophen). Most people have few or non-specific symptoms in the first 24 hours following overdose. These include feeling tired, abdominal pain, or nausea. This is typically followed by a couple of days without any symptoms, after which yellowish skin, blood clotting problems, an

## Step 8: Defining the QLoRA configuration

In [12]:
dtype_computation = getattr(torch, compute_dtype_bnb)

bnb_setup = None
if enable_4bit:
    bnb_setup = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=quant_type_bnb,
        bnb_4bit_use_double_quant=double_quant_flag,
        bnb_4bit_compute_dtype=dtype_computation,
    )

print("BitsAndBytes config:", bnb_setup)

BitsAndBytes config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## Step 9: Loading the pre-trained LLaMA 2 model

In [13]:
llama_model = AutoModelForCausalLM.from_pretrained(
    model_identifier,
    quantization_config=bnb_setup,
    device_map=device_assignment,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# Required for gradient checkpointing and PEFT/QLoRA training stability.
llama_model.config.use_cache = False

if enable_4bit:
    llama_model = prepare_model_for_kbit_training(llama_model)

print("Model loaded:", model_identifier)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## Step 10: Loading the pre-trained tokenizer for the LLaMA 2 model

In [14]:
llama_tokenizer = AutoTokenizer.from_pretrained(model_identifier, use_fast=True)

if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("pad_token:", llama_tokenizer.pad_token)
print("eos_token:", llama_tokenizer.eos_token)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer loaded.
pad_token: </s>
eos_token: </s>


## Step 11: Setting up the configuration for the LoRA fine-tuning method

In [15]:
peft_setup = LoraConfig(
    lora_alpha=lora_hyper_alpha,
    lora_dropout=lora_hyper_dropout,
    r=lora_hyper_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print(peft_setup)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'up_proj', 'q_proj', 'o_proj', 'k_proj', 'v_proj', 'down_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## Step 12: Creating a training configuration by setting the training parameters

In [16]:
train_args = SFTConfig(
    output_dir=results_dir,
    num_train_epochs=epochs_count,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=accumulation_steps,
    learning_rate=train_learning_rate,
    weight_decay=decay_rate,
    optim=optimizer_type,
    save_strategy="no",
    logging_steps=log_interval,
    fp16=enable_fp16,
    bf16=enable_bf16,
    max_grad_norm=grad_norm_limit,
    max_steps=steps_limit,
    warmup_ratio=warmup_percentage,
    group_by_length=length_grouping,
    lr_scheduler_type=scheduler_type,
    gradient_checkpointing=checkpointing_flag,
    report_to=[],
    dataset_text_field="text",
    max_length=sequence_length_max,
    packing=enable_packing,
)

print(train_args)

SFTConfig(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=True,
do_eval=False,
do_predict=False,
do_train=False,
eos_token=

## Step 13: Creating the Supervised Fine-Tuning Trainer

In [17]:
llama_sftt_trainer = SFTTrainer(
    model=llama_model,
    args=train_args,
    train_dataset=training_data,
    processing_class=llama_tokenizer,
    peft_config=peft_setup,
)

print("SFTTrainer created successfully.")

Adding EOS to train dataset:   0%|          | 0/6861 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6861 [00:00<?, ? examples/s]

SFTTrainer created successfully.


## Step 14: Training the model

In [18]:
# Optional sanity check before training.
# This verifies that tokenization works and the trainer can build a batch.

sample_batch = next(iter(llama_sftt_trainer.get_train_dataloader()))
print(sample_batch.keys())
print("input_ids shape:", sample_batch["input_ids"].shape)
print("labels shape:", sample_batch["labels"].shape)


dict_keys(['input_ids', 'labels', 'attention_mask'])
input_ids shape: torch.Size([1, 512])
labels shape: torch.Size([1, 512])


In [19]:
training_result = llama_sftt_trainer.train()
print(training_result)

# Save only the LoRA adapter. This is lightweight and is the recommended output for PEFT/QLoRA.
llama_sftt_trainer.model.save_pretrained("./tinyllama-medical-lora-adapter")
llama_tokenizer.save_pretrained("./tinyllama-medical-lora-adapter")

print("LoRA adapter saved to ./tinyllama-medical-lora-adapter")



{'loss': 2.0923, 'grad_norm': 0.75390625, 'learning_rate': 0.00019476531711828027, 'entropy': 2.0149457097053527, 'num_tokens': 10240.0, 'mean_token_accuracy': 0.5496086120605469, 'epoch': 0.002915026963999417}
{'loss': 1.633, 'grad_norm': 1.0390625, 'learning_rate': 0.00016473862847818277, 'entropy': 1.7681267380714416, 'num_tokens': 20480.0, 'mean_token_accuracy': 0.6200587064027786, 'epoch': 0.005830053927998834}
{'loss': 1.3468, 'grad_norm': 0.6796875, 'learning_rate': 0.0001161781996552765, 'entropy': 1.3754882097244263, 'num_tokens': 30720.0, 'mean_token_accuracy': 0.7001956880092621, 'epoch': 0.00874508089199825}
{'loss': 1.3362, 'grad_norm': 0.71484375, 'learning_rate': 6.298618446600856e-05, 'entropy': 1.29599387049675, 'num_tokens': 40960.0, 'mean_token_accuracy': 0.7026418715715408, 'epoch': 0.011660107855997668}
{'loss': 1.3459, 'grad_norm': 0.57421875, 'learning_rate': 2.0390693429435627e-05, 'entropy': 1.3209435582160949, 'num_tokens': 51200.0, 'mean_token_accuracy': 0.70

## Step 15: Chatting with the model

In [20]:
# Important: the training dataset uses Llama-2 [INST] formatting.
# Therefore, inference should use the same [INST] format instead of TinyLlama's default chat template.

import torch

def format_llama2_style_prompt(question):
    system_prompt = (
        "You are a helpful, respectful, and honest medical education assistant. "
        "Answer clearly and safely using general educational information."
    )
    return f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{question.strip()} [/INST]"

def generate_medical_answer(question, max_new_tokens=220):
    prompt = format_llama2_style_prompt(question)

    llama_sftt_trainer.model.eval()

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=512,
    )

    device = next(llama_sftt_trainer.model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output_ids = llama_sftt_trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=llama_tokenizer.eos_token_id,
            eos_token_id=llama_tokenizer.eos_token_id,
            use_cache=False,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    answer = llama_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return answer

user_prompt = "Please tell me about bursitis."
print(generate_medical_answer(user_prompt))

Bursitis is an inflammation of the bursa (sacks) in any part of the body. Bursitis can occur in various parts of the body and not necessarily involving the joints. It is a common problem among people who spend a lot of time sitting or standing on their feet. It may be caused by injury, prolonged immobilization, or poor posture. It can also result from certain medications, such as corticosteroids, NSAIDs, and other anti-inflammatory drugs.
The causes of bursitis are multifactorial and include:

1. Injury to the affected area
2. Stress from prolonged sitting
3. Long-term immobility due to chronic pain or illness
4. Poor posture or incorrect body positioning
5. Poor nutrition
6. Genetics
7. Immune system disorders 
8. Other unknown factors

Symptoms
Symptoms of burs


In [21]:
# Optional: interactive helper function for testing the fine-tuned model.
# This uses the same prompt format as the fine-tuning dataset.

def ask_finetuned_model(question, max_new_tokens=220):
    answer = generate_medical_answer(question, max_new_tokens=max_new_tokens)

    if len(answer.strip()) <= 3:
        return "The model did not generate a useful response. Try rephrasing your question."

    return answer